**Author:** Dorys Trujillo  
**Project:** Legal Uncertainty Index (IIJ)   
**Data Source:** Ministry of Commerce, Industry and Tourism (MinCIT)  
**Period:** 2018–2025

## Natural Language Preprocessing

In [7]:
# Imports
from __future__ import annotations

import json
import re
import unicodedata
from pathlib import Path
from typing import Iterable

import pandas as pd
import spacy
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer

In [13]:
# define project root dynamically (assumes execution inside notebooks/)
PROJECT_ROOT = Path().resolve().parent

# input: corpus organized by regulation
OCR_ROOT = PROJECT_ROOT / "data_processed" / "txt_normative_rag"

# output: cleaned corpus
CLEAN_ROOT = PROJECT_ROOT / "data_processed" / "corpus_clean"

# manifests: global tracking files
MANIFEST_ROOT = PROJECT_ROOT / "manifests"

# create output directories if they do not exist
CLEAN_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

# sanity checks
#print("project_root:", PROJECT_ROOT)
#print("ocr_root exists:", OCR_ROOT.exists(), OCR_ROOT)
#print("clean_root:", CLEAN_ROOT)
#print("manifest_root:", MANIFEST_ROOT)

# discover all txt files recursively (preserving folder structure by regulation)
ocr_files = sorted(OCR_ROOT.rglob("*.txt"))

print(f"ocr files found: {len(ocr_files)}")

# We build basic inventory
rows = []
for txt_path in ocr_files:
    rel_path = txt_path.relative_to(OCR_ROOT)
    content = txt_path.read_text(encoding="utf-8", errors="ignore")
    
    rows.append({
        "ocr_file": str(txt_path),
        "rel_path": str(rel_path),
        "n_chars": len(content),
        "n_lines": content.count("\n") + 1,
        "is_empty": len(content.strip()) == 0
    })

df_ocr_inventory = pd.DataFrame(rows)

print("\nQuick summary:")
print(df_ocr_inventory["is_empty"].value_counts(dropna=False))

display(df_ocr_inventory.head(10))

ocr files found: 334

Quick summary:
is_empty
False    334
Name: count, dtype: int64


,ocr_file,rel_path,n_chars,n_lines,is_empty
0,C:\Users\dtruj\Documentos\proyectos\legal-unce...,Aplicación de derechos compensatorios\id_0085.txt,83069,1287,False
1,C:\Users\dtruj\Documentos\proyectos\legal-unce...,Arancel de Aduanas\id_0008.txt,3116,58,False
2,C:\Users\dtruj\Documentos\proyectos\legal-unce...,Arancel de Aduanas\id_0019.txt,4460,76,False
3,C:\Users\dtruj\Documentos\proyectos\legal-unce...,Arancel de Aduanas\id_0026.txt,4947,120,False
4,C:\Users\dtruj\Documentos\proyectos\legal-unce...,Arancel de Aduanas\id_0030.txt,2482,53,False
5,C:\Users\dtruj\Documentos\proyectos\legal-unce...,Arancel de Aduanas\id_0039.txt,5116,104,False
6,C:\Users\dtruj\Documentos\proyectos\legal-unce...,Arancel de Aduanas\id_0050.txt,4414,66,False
7,C:\Users\dtruj\Documentos\proyectos\legal-unce...,Arancel de Aduanas\id_0051.txt,5963,116,False
8,C:\Users\dtruj\Documentos\proyectos\legal-unce...,Arancel de Aduanas\id_0055.txt,6458,131,False
9,C:\Users\dtruj\Documentos\proyectos\legal-unce...,Arancel de Aduanas\id_0067.txt,11020,247,False


#### 1. Basic Cleaning Stage
This step performs basic text preprocessing on the OCR corpus. It cleans and normalizes the text (line breaks, spaces, Unicode, and hyphenated words), preserves the original folder structure by regulation, saves the cleaned files in a new processing stage, and generates a manifest to track transformations and ensure traceability.

In [15]:
# subfolder for this stage
BASIC_CLEAN_ROOT = CLEAN_ROOT / "basic_clean"
BASIC_CLEAN_ROOT.mkdir(parents=True, exist_ok=True)

BASIC_MANIFEST_PATH = MANIFEST_ROOT / "manifest_basic_clean.csv"

# basic text cleaning function
def basic_clean_text(text: str) -> str:
    # 1) normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # 2) replace invisible characters and unusual spaces
    text = text.replace("\ufeff", " ")
    text = text.replace("\xa0", " ")
    text = text.replace("\x0c", "\n")

    # 3) replace common typographic ligatures
    text = text.replace("ﬁ", "fi")
    text = text.replace("ﬂ", "fl")

    # 4) normalize unicode
    text = unicodedata.normalize("NFKC", text)

    # 5) join words split by hyphen at line breaks
    # example: econo-\nmía -> economía
    text = re.sub(r"(\w+)-\n(\w+)", r"\1\2", text)

    # 6) collapse repeated spaces and tabs
    text = re.sub(r"[ \t]+", " ", text)

    # 7) trim spaces at the beginning and end of lines
    text = re.sub(r"[ \t]*\n[ \t]*", "\n", text)

    # 8) reduce multiple blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # 9) final strip
    text = text.strip()

    return text

# process all txt files
rows = []

for txt_path in ocr_files:
    rel_path = txt_path.relative_to(OCR_ROOT)
    out_path = BASIC_CLEAN_ROOT / rel_path
    out_path.parent.mkdir(parents=True, exist_ok=True)

    raw_text = txt_path.read_text(encoding="utf-8", errors="ignore")
    clean_text = basic_clean_text(raw_text)

    out_path.write_text(clean_text, encoding="utf-8")

    rows.append({
        "input_file": str(txt_path),
        "output_file": str(out_path),
        "rel_path": str(rel_path),
        "raw_chars": len(raw_text),
        "clean_chars": len(clean_text),
        "char_diff": len(raw_text) - len(clean_text),
        "raw_lines": raw_text.count("\n") + 1,
        "clean_lines": clean_text.count("\n") + 1,
        "is_empty_after_clean": len(clean_text.strip()) == 0,
        "status": "ok"
    })

df_basic_clean = pd.DataFrame(rows)
df_basic_clean.to_csv(BASIC_MANIFEST_PATH, index=False, encoding="utf-8")

print("files processed:", len(df_basic_clean))
print("empty after cleaning:", df_basic_clean["is_empty_after_clean"].sum())
print("manifest saved to:", BASIC_MANIFEST_PATH)

#display(df_basic_clean.head(10))

files processed: 334
empty after cleaning: 0
manifest saved to: C:\Users\dtruj\Documentos\proyectos\legal-uncertainty-index\manifests\manifest_basic_clean.csv


In [17]:
#Check that the character reduction is reasonable
#df_basic_clean[["raw_chars", "clean_chars", "char_diff"]].describe()

### 2. Structural Normalization

This stage performs structural normalization of the cleaned corpus by identifying and extracting the most relevant legal content. It detects key sections such as the decree title, action clauses (e.g., “decreta”), and the first article, applying a prioritized strategy to retain meaningful text. The process reduces noise, standardizes formatting, preserves the original folder structure, and generates a manifest to track transformations and applied strategies.

In [18]:
# subfolder for this stage
NORMALIZED_ROOT = CLEAN_ROOT / "normalized"
NORMALIZED_ROOT.mkdir(parents=True, exist_ok=True)

NORMALIZED_MANIFEST_PATH = MANIFEST_ROOT / "manifest_normalized.csv"

# structural patterns
TITLE_START_PATTERNS = [
    r"por el cual",
    r"por la cual",
    r"por medio del cual",
    r"por medio de la cual",
]

PRESIDENT_MARKERS = [
    r"el presidente de la república",
    r"el presidente de la republica",
    r"la ministra de",
    r"el ministro de",
]

ACTION_PATTERNS = [
    r"\bdecreta\s*:?\b",
    r"\bresuelve\s*:?\b",
]

ARTICLE_START_PATTERNS = [
    r"art[ií]culo\s+1\b",
    r"art[ií]culo\s+primero\b",
]

# compile regex patterns once
TITLE_START_REGEX = [re.compile(pattern, flags=re.IGNORECASE) for pattern in TITLE_START_PATTERNS]
PRESIDENT_REGEX = [re.compile(pattern, flags=re.IGNORECASE) for pattern in PRESIDENT_MARKERS]
ACTION_REGEX = [re.compile(pattern, flags=re.IGNORECASE) for pattern in ACTION_PATTERNS]
ARTICLE_START_REGEX = [re.compile(pattern, flags=re.IGNORECASE) for pattern in ARTICLE_START_PATTERNS]


def find_first_match_position(text: str, patterns: list[re.Pattern], start_pos: int = 0):
    """return the first match position among multiple compiled patterns."""
    tail = text[start_pos:]
    positions = []

    for pattern in patterns:
        match = pattern.search(tail)
        if match:
            positions.append(start_pos + match.start())

    return min(positions) if positions else None


def find_title_start(text: str, search_window: int = 4000):
    """find the beginning of the decree title within the first part of the document."""
    head = text[:search_window]
    positions = []

    for pattern in TITLE_START_REGEX:
        match = pattern.search(head)
        if match:
            positions.append(match.start())

    return min(positions) if positions else None


def extract_title_block(text: str):
    """extract the decree title block using title, authority, and action markers."""
    start = find_title_start(text)
    if start is None:
        return None

    president_pos = find_first_match_position(text, PRESIDENT_REGEX, start_pos=start)
    action_pos = find_first_match_position(text, ACTION_REGEX)

    candidates = [pos for pos in [president_pos, action_pos] if pos is not None and pos > start]

    if candidates:
        end = min(candidates)
        title = text[start:end].strip()
    else:
        title = text[start:start + 1200].strip()

    title = title.strip('“”"\' \n\t')
    title = re.sub(r"\s+", " ", title)

    return title if len(title) > 20 else None


def normalize_text_structural(text: str):
    """apply structural normalization and keep the most relevant legal content."""
    text = text.lower()
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    title_block = extract_title_block(text)
    action_pos = find_first_match_position(text, ACTION_REGEX)
    article_pos = find_first_match_position(text, ARTICLE_START_REGEX)

    # priority 1: title block + action block
    if title_block and action_pos is not None:
        action_block = text[action_pos:].strip()
        text = f"{title_block}\n\n{action_block}"
        strategy = "title_block_plus_action"

    # priority 2: title block + first article
    elif title_block and article_pos is not None:
        article_block = text[article_pos:].strip()
        text = f"{title_block}\n\n{article_block}"
        strategy = "title_block_plus_article"

    # fallback 1: action block only
    elif action_pos is not None:
        text = text[action_pos:].strip()
        strategy = "action_only"

    # fallback 2: first article only
    elif article_pos is not None:
        text = text[article_pos:].strip()
        strategy = "article_only"

    # fallback 3: no structural cut
    else:
        strategy = "fallback_no_structural_cut"

    # residual cleanup
    text = re.sub(r"\b(página|pagina)\s*\d+\b", " ", text)
    text = re.sub(r"-\s*\d+\s*-", " ", text)
    text = re.sub(r"#{2,}\s*page\s*\d+", " ", text)

    lines = text.split("\n")
    lines = [line.strip() for line in lines if len(line.strip()) > 3]
    text = "\n".join(lines)

    text = re.sub(r" +", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip(), strategy


# process files from basic cleaning stage
rows = []

basic_files = sorted(BASIC_CLEAN_ROOT.rglob("*.txt"))

for txt_path in basic_files:
    rel_path = txt_path.relative_to(BASIC_CLEAN_ROOT)
    out_path = NORMALIZED_ROOT / rel_path
    out_path.parent.mkdir(parents=True, exist_ok=True)

    text = txt_path.read_text(encoding="utf-8", errors="ignore")
    norm_text, strategy = normalize_text_structural(text)

    out_path.write_text(norm_text, encoding="utf-8")

    rows.append({
        "input_file": str(txt_path),
        "output_file": str(out_path),
        "rel_path": str(rel_path),
        "chars_before": len(text),
        "chars_after": len(norm_text),
        "char_diff": len(text) - len(norm_text),
        "lines_before": text.count("\n") + 1,
        "lines_after": norm_text.count("\n") + 1,
        "strategy": strategy,
        "status": "ok"
    })

df_normalized = pd.DataFrame(rows)
df_normalized.to_csv(NORMALIZED_MANIFEST_PATH, index=False, encoding="utf-8")

print("files processed:", len(df_normalized))
print("manifest saved to:", NORMALIZED_MANIFEST_PATH)
print("\nstrategies applied:")
print(df_normalized["strategy"].value_counts())

display(df_normalized[["chars_before", "chars_after", "char_diff"]].describe())

files processed: 334
manifest saved to: C:\Users\dtruj\Documentos\proyectos\legal-uncertainty-index\manifests\manifest_normalized.csv

strategies applied:
strategy
title_block_plus_action       332
fallback_no_structural_cut      1
title_block_plus_article        1
Name: count, dtype: int64


,chars_before,chars_after,char_diff
count,334.000000,334.000000,334.000000
mean,18749.889222,13059.682635,5690.206587
std,22921.447533,22400.867607,3737.758544
min,1443.000000,487.000000,723.000000
25%,6205.000000,1795.250000,3443.250000
50%,10576.500000,4331.000000,4892.500000
75%,21442.000000,13747.000000,7033.000000
max,150286.000000,146256.000000,35141.000000
